# Spacing Statistics

## 1. Importing / Installing Packages

In [1]:
import os # Importing os module for operating system dependent functionality

import pandas as pd # Importing pandas package

# Set the maximum number of columns to display to None
pd.set_option('display.max_columns', None)

import numpy as np # Importing numpy package

from typing import Dict, Tuple, List, Union, Optional, ClassVar # Importing specific types from typing module

from src.utils import DatabricksOdbcConnector # Importing DatabricksOdbcConnector class from database_manager module

from tqdm import tqdm # Importing tqdm for progress bar functionality

from joblib import Parallel, delayed # Importing Parallel and delayed for parallel processing

from matplotlib import pyplot as plt # Importing pyplot from matplotlib for plotting

from pyproj import Geod # Importing Geod class from pyproj for geodetic calculations

# Setting matplotlib to inline mode for Jupyter notebooks
%matplotlib inline

%config InlineBackend.figure_format = 'svg' # Configuring inline backend to use SVG format for figures

from src.well_data import WellDataLoader, GeoSurveyProcessor # Importing custom classes for well data management

# from src.utils import reorder_columns # Importing utility function to reorder DataFrame columns

## 2. Defining Functions

### 2.1. Defining Functions that is used in calculation for i-k pair dataframe and Spacing Stats

In [2]:
class WellSpacingCalculator:
    """
    Class for calculating well spacing metrics and directional relationships using
    3D lateral midpoint alignment and curvature-aware distances.
    Midpoints are projected in 2D space to remove lateral-length bias when calculating spacing.
    """

    # Class-level constant (shared, immutable-by-convention)
    _DIR8_LABELS: ClassVar[np.ndarray] = np.array(
        ["E","NE","N","NW","W","SW","S","SE"], dtype=object
    )

    def __init__(self, trajectories: Union[Dict[str, pd.DataFrame], pd.DataFrame]):
        if isinstance(trajectories, pd.DataFrame):
            if "uwi" not in trajectories.columns:
                raise ValueError("Trajectory DataFrame must contain 'uwi' column.")
            self._trajectory_df = trajectories.reset_index(drop=True)
            self.trajectories = {
                cid: group for cid, group in self._trajectory_df.groupby("uwi")
            }
        elif isinstance(trajectories, dict):
            self.trajectories = trajectories
            self._trajectory_df = pd.concat(
                trajectories.values(), keys=trajectories.keys()
            ).reset_index(drop=True)
        else:
            raise ValueError("Invalid type for trajectories. Must be DataFrame or Dict.")

    def _calculate_spacing_statistics(
        self,
        frac: float = 0.5,
        batch_size: int = 1_000_000,
        max_distance_miles: Optional[float] = 20.0,
        save_batches_dir: Optional[str] = None,
        use_interpolation: bool = True,
        *,
        # new knobs
        step_ft: int = 100,
        n_samples: Optional[int] = None,
        max_crossline_ft: Optional[float] = 900.0,
        crossline_percentile: float = 5.0,
        ds_crossline_step_ft: int = 300,
        emit_rejected: bool = False,
        use_pca_axis: bool = True,
    ) -> Optional[pd.DataFrame]:
        """
        Compute spacing distances between well pairs using the overlap → clip → sample method.
        Produces horizontal (mean |Δx| with Y-alignment), vertical (from midpoints), 3D,
        and geodetic 8-way modal direction across the overlapped segment.
        """
        # Build the pair cache once (vectorized coarse arrays + local frames)
        self._build_pair_cache(
            use_pca_axis=use_pca_axis,
            ds_crossline_step_ft=ds_crossline_step_ft,
        )

        # 1) Midpoints + drill dirs (vertical uses these)
        midpoint_df = self._compute_normalized_midpoints(frac=frac, use_interpolation=use_interpolation)
        drill_dirs = self._compute_drill_directions()
        midpoint_df["drill_direction"] = drill_dirs

        # 2) Arrays
        ids = midpoint_df.index.to_numpy()
        coords = midpoint_df[["x", "y", "tvd"]].to_numpy()
        lat_lon = midpoint_df[["latitude", "longitude"]].to_numpy()
        directions = midpoint_df["drill_direction"].to_numpy()

        # 3) Candidate pairs
        if max_distance_miles is not None:
            lat = lat_lon[:, 0]
            lon = lat_lon[:, 1]
            i_idx, k_idx = self._filter_close_pairs(lat, lon, max_distance_miles)
        else:
            i_idx, k_idx = self._get_pairwise_indices(ids)

        pairs = list(zip(i_idx, k_idx))
        batch_generator = list(self._batch_filtered_indices(pairs, batch_size=batch_size))
        n_batches = len(batch_generator)

        if save_batches_dir:
            os.makedirs(save_batches_dir, exist_ok=True)

        def process_and_save(batch_number: int, i_idx: np.ndarray, k_idx: np.ndarray):
            batch_df = self._process_batch(
                i_idx, k_idx, ids, coords, directions,
                step_ft=step_ft, n_samples=n_samples,
                max_crossline_ft=max_crossline_ft,
                crossline_percentile=crossline_percentile,
                ds_crossline_step_ft=ds_crossline_step_ft,
                emit_rejected=emit_rejected, use_pca_axis=use_pca_axis,
            )
            if save_batches_dir:
                filepath = os.path.join(save_batches_dir, f"spacing_batch_{batch_number:04d}.parquet")
                batch_df.to_parquet(filepath, index=False)
            return batch_df

        tqdm_kwargs = {
            "desc": "🚀 Calculating Spacing (Parallel)",
            "dynamic_ncols": True,
            "smoothing": 0.3,
            "bar_format": "{desc}: |{bar:40}| {percentage:3.0f}% {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
            "ascii": "░▒█",
            "leave": True,
        }

        results = Parallel(n_jobs=-1)(
            delayed(process_and_save)(batch_num, i_idx, k_idx)
            for batch_num, (i_idx, k_idx) in tqdm(enumerate(batch_generator), total=n_batches, **tqdm_kwargs)
        )

        if save_batches_dir:
            print(f"✅ All batches saved to {save_batches_dir}")
            return None
        return pd.concat(results, ignore_index=True) if results else pd.DataFrame()
    
    def _load_saved_batches(self, batch_folder: str) -> pd.DataFrame:
        """
        Load all saved spacing batch Parquet files from a folder and combine into a single DataFrame.

        Parameters
        ----------
        batch_folder : str
            Path to the folder where batch Parquet files are stored.

        Returns
        -------
        pd.DataFrame
            Combined spacing DataFrame.
        """
        if not os.path.isdir(batch_folder):
            raise FileNotFoundError(f"Batch folder '{batch_folder}' not found.")

        batch_files = sorted([
            os.path.join(batch_folder, f)
            for f in os.listdir(batch_folder)
            if f.endswith(".parquet")
        ])

        if not batch_files:
            raise ValueError(f"No Parquet files found in folder '{batch_folder}'.")

        print(f"🔍 Found {len(batch_files)} batch files. Loading and combining...")

        dfs = []
        for file in batch_files:
            dfs.append(pd.read_parquet(file))

        combined_df = pd.concat(dfs, ignore_index=True)
        print(f"✅ Loaded {len(combined_df):,} rows from all batches.")
        return combined_df

    def _filter_close_pairs(self, lat: np.ndarray, lon: np.ndarray, max_distance_miles: float = 20.0) -> Tuple[np.ndarray, np.ndarray]:

        lat1, lat2 = np.meshgrid(lat, lat, indexing="ij")
        lon1, lon2 = np.meshgrid(lon, lon, indexing="ij")

        delta_lat = np.abs(lat1 - lat2)
        delta_lon = np.abs(lon1 - lon2)

        miles_per_lat_degree = 69.0
        miles_per_lon_degree = 69.0 * np.cos(np.radians(lat))
        miles_per_lon_degree_matrix = np.add.outer(miles_per_lon_degree, miles_per_lon_degree) / 2.0

        rough_dist_miles = np.sqrt(
            (delta_lat * miles_per_lat_degree)**2 + (delta_lon * miles_per_lon_degree_matrix)**2
        )

        mask = (rough_dist_miles <= max_distance_miles) & (delta_lat + delta_lon > 0)
        i_idx, k_idx = np.where(mask)

        return i_idx, k_idx

    def _get_pairwise_indices(self, uwis: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate all valid pairwise (i, k) UWI combinations from an array of well IDs,
        excluding self-comparisons (i != k).

        Parameters
        ----------
        uwis : np.ndarray
            Array of unique well identifiers.

        Returns
        -------
        Tuple[np.ndarray, np.ndarray]
            Two 1D arrays of i and k UWIs representing all valid (i, k) pairs.
        """
        # Generate meshgrid of all possible UWI pairs
        n = len(uwis)
        i_idx, k_idx = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
            
        # Exclude self-comparisons (where i_uwi == k_uwi)
        valid_mask = i_idx != k_idx

        return i_idx[valid_mask], k_idx[valid_mask]

    def _batch_filtered_indices(self, pairs: List[Tuple[int, int]], batch_size: int = 1_000_000):
        """
        Vectorized batching of prefiltered well pairs.

        Parameters
        ----------
        pairs : List[Tuple[int, int]]
            List of (i_idx, k_idx) pairs.
        batch_size : int
            Number of pairs per batch.

        Yields
        ------
        Tuple[np.ndarray, np.ndarray]
            i_idx and k_idx arrays for each batch.
        """
        pairs_array = np.array(pairs)  # Convert list of tuples directly to 2D array (N, 2)
        n_pairs = pairs_array.shape[0]

        # Vectorized slicing
        split_indices = np.arange(0, n_pairs, batch_size)

        for start_idx in split_indices:
            end_idx = min(start_idx + batch_size, n_pairs)
            batch = pairs_array[start_idx:end_idx]
            yield batch[:, 0], batch[:, 1]

    def _compute_normalized_midpoints(self, frac: float = 0.5, use_interpolation: bool = True) -> pd.DataFrame:
        """
        Computes midpoints for each well either by interpolating along the well trajectory
        using MD-based fractional position or by averaging heel and toe coordinates.

        Parameters:
        -----------
        frac : float
            Fractional location along the lateral to compute the midpoint (0.0 to 1.0).
        use_interpolation : bool
            If True, uses curvature-aware interpolation along MD.
            If False, uses geometric midpoint between heel and toe.

        Returns:
        --------
        pd.DataFrame indexed by 'uwi', containing:
            ['x', 'y', 'tvd', 'latitude', 'longitude']
        """
        df = self._trajectory_df.copy()
        df = df.sort_values(["uwi", "md"]).reset_index(drop=True)

        if not use_interpolation:
            # Simple geometric midpoint (fast)
            heel_toe_df = (
                df.groupby("uwi")
                .agg(
                    heel_x=("x", "first"),
                    heel_y=("y", "first"),
                    heel_tvd=("tvd", "first"),
                    heel_lat=("latitude", "first"),
                    heel_lon=("longitude", "first"),
                    toe_x=("x", "last"),
                    toe_y=("y", "last"),
                    toe_tvd=("tvd", "last"),
                    toe_lat=("latitude", "last"),
                    toe_lon=("longitude", "last"),
                )
            )

            midpoint_df = pd.DataFrame({
                "x": (heel_toe_df["heel_x"] + heel_toe_df["toe_x"]) / 2,
                "y": (heel_toe_df["heel_y"] + heel_toe_df["toe_y"]) / 2,
                "tvd": (heel_toe_df["heel_tvd"] + heel_toe_df["toe_tvd"]) / 2,
                "latitude": (heel_toe_df["heel_lat"] + heel_toe_df["toe_lat"]) / 2,
                "longitude": (heel_toe_df["heel_lon"] + heel_toe_df["toe_lon"]) / 2,
            })
            midpoint_df.index.name = "uwi"
            return midpoint_df

        # Interpolated midpoint (MD-based)
        min_md = df.groupby("uwi")["md"].transform("min")
        max_md = df.groupby("uwi")["md"].transform("max")
        df["normalized_md"] = (df["md"] - min_md) / (max_md - min_md)

        df["row_index"] = df.groupby("uwi").cumcount()
        df["prev_idx"] = df.groupby("uwi")["normalized_md"].transform(lambda x: x.searchsorted(frac, side="right") - 1)
        df["next_idx"] = df["prev_idx"] + 1
        df["next_idx"] = np.minimum(df["next_idx"], df["row_index"].groupby(df["uwi"]).transform("max"))

        df_prev = df.groupby("uwi").apply(lambda g: g.loc[g["row_index"] == g["prev_idx"].iloc[0]]).reset_index(drop=True)
        df_next = df.groupby("uwi").apply(lambda g: g.loc[g["row_index"] == g["next_idx"].iloc[0]]).reset_index(drop=True)

        merged = pd.merge(df_prev, df_next, on="uwi", suffixes=("_prev", "_next"))

        delta = merged["normalized_md_next"] - merged["normalized_md_prev"]
        delta = delta.replace(0, np.nan)
        ratio = (frac - merged["normalized_md_prev"]) / delta

        midpoint_df = pd.DataFrame({
            "x": merged["x_prev"] + ratio * (merged["x_next"] - merged["x_prev"]),
            "y": merged["y_prev"] + ratio * (merged["y_next"] - merged["y_prev"]),
            "tvd": merged["tvd_prev"] + ratio * (merged["tvd_next"] - merged["tvd_prev"]),
            "latitude": merged["latitude_prev"] + ratio * (merged["latitude_next"] - merged["latitude_prev"]),
            "longitude": merged["longitude_prev"] + ratio * (merged["longitude_next"] - merged["longitude_prev"]),
        })
        midpoint_df["uwi"] = merged["uwi"]
        return midpoint_df.set_index("uwi")
    
    def _compute_drill_directions(self) -> pd.Series:
        median_azimuth = self._trajectory_df.groupby("uwi")["azimuth"].median()
        is_ew = ((median_azimuth >= 45) & (median_azimuth <= 135)) | ((median_azimuth >= 225) & (median_azimuth <= 315))
        return pd.Series(np.where(is_ew, "EW", "NS"), index=median_azimuth.index, name="drill_direction")
    
    def _build_pair_cache(
        self,
        use_pca_axis: bool,
        ds_crossline_step_ft: int,
        ds_min_points: int = 16,
        ds_max_points: int = 64,
    ):
        """
        Precompute per-well:
        - origin, ex, ey (local frame metadata)
        - coarse resample XY (same length across wells) for vectorized cross-line precheck
        """
        # Compute each well's lateral length to pick a *global* coarse point count
        lengths = {}
        for uwi, df in self.trajectories.items():
            XY = df.sort_values("md")[["x","y"]].to_numpy()
            d = np.hypot(np.diff(XY[:,0]), np.diff(XY[:,1]))
            lengths[uwi] = float(d.sum())

        # Choose a *single* coarse count using median length / step, then clamp
        median_len = np.median(list(lengths.values())) if lengths else 3000.0
        m_guess = int(np.ceil(max(median_len, 1.0) / max(ds_crossline_step_ft, 1))) + 1
        M_ds = int(np.clip(m_guess, ds_min_points, ds_max_points))

        cache = {
            "origin": {},
            "ex": {},
            "ey": {},
            "XY_coarse": {},
            "M_ds": M_ds,
            "use_pca_axis": use_pca_axis,
        }

        for uwi, df in self.trajectories.items():
            df = df.sort_values("md")
            XY = df[["x","y"]].to_numpy()

            # local frame metadata
            pts = XY
            heel, toe = pts[0], pts[-1]
            origin = heel.copy()
            if use_pca_axis:
                C = pts - pts.mean(0)
                _, _, Vt = np.linalg.svd(C, full_matrices=False)
                ex = Vt[0]
                if np.dot(ex, toe - heel) < 0:
                    ex = -ex
            else:
                v = toe - heel
                ex = v / (np.linalg.norm(v) + 1e-12)
            ey = np.array([-ex[1], ex[0]])

            cache["origin"][uwi] = origin
            cache["ex"][uwi] = ex
            cache["ey"][uwi] = ey

            # coarse resample to common length M_ds by arclength
            d = np.hypot(np.diff(XY[:,0]), np.diff(XY[:,1]))
            s = np.concatenate([[0.0], np.cumsum(d)])
            L = s[-1] if s.size else 0.0
            if L <= 0.0:
                XYc = np.repeat(XY[:1], M_ds, axis=0)
            else:
                t = np.linspace(0.0, 1.0, M_ds)
                sx = np.interp(t*L, s, XY[:,0])
                sy = np.interp(t*L, s, XY[:,1])
                XYc = np.column_stack([sx, sy])

            cache["XY_coarse"][uwi] = XYc

        self._paircache = cache

    def _build_local_frame_from_i(self, df_i: pd.DataFrame, use_pca_axis: bool = True
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Build well_i's local frame from its UTM x,y points.
        origin: heel (first by MD); ex: along-lateral (PCA or heel→toe); ey: 90° CCW from ex.
        """
        pts = df_i.sort_values("md")[["x","y"]].to_numpy()
        heel, toe = pts[0], pts[-1]
        origin = heel.copy()

        if use_pca_axis:
            C = pts - pts.mean(0)
            _, _, Vt = np.linalg.svd(C, full_matrices=False)
            ex = Vt[0]
            if np.dot(ex, toe - heel) < 0:
                ex = -ex
        else:
            v = toe - heel
            ex = v / (np.linalg.norm(v) + 1e-12)

        ey = np.array([-ex[1], ex[0]])
        return origin, ex, ey
    
    def _project_xy_to_frame(self, df: pd.DataFrame, origin: np.ndarray, ex: np.ndarray, ey: np.ndarray
    ) -> np.ndarray:
        """Return Nx2 array of (x_local, y_local) from UTM x,y."""
        XY = df[["x","y"]].to_numpy()
        R = XY - origin
        return np.column_stack([R @ ex, R @ ey])
    
    def _clip_polyline_by_x_band(self, X: np.ndarray, band: Tuple[float, float]) -> np.ndarray:
        """
        Clip polyline X[:,0]=x, X[:,1]=y to x ∈ [x_lo, x_hi] and INSERT boundary
        intersection points by linear interpolation. Guarantees ≥2 points if the
        band intersects the polyline.
        """
        x_lo, x_hi = band
        x = X[:, 0]
        pts = []

        for j in range(len(X) - 1):
            x0, x1 = x[j], x[j + 1]
            P0, P1 = X[j], X[j + 1]

            seg_min, seg_max = (x0, x1) if x0 <= x1 else (x1, x0)
            if seg_max < x_lo or seg_min > x_hi:
                continue  # segment outside band

            # keep start if inside
            if x_lo <= x0 <= x_hi:
                pts.append(P0)

            # intersections with boundaries (strict crossing)
            for xb in (x_lo, x_hi):
                denom = (x1 - x0)
                if denom != 0.0 and (x0 - xb) * (x1 - xb) < 0.0:
                    t = (xb - x0) / denom
                    pts.append(P0 + t * (P1 - P0))

            # keep end if last seg and inside
            if j == len(X) - 2 and (x_lo <= x1 <= x_hi):
                pts.append(P1)

        if not pts:
            return np.empty((0, 2))

        P = np.vstack(pts)
        # drop exact duplicates while preserving order
        keep = np.ones(len(P), dtype=bool)
        if len(P) > 1:
            dup = np.all(np.isclose(np.diff(P, axis=0), 0.0, atol=1e-9), axis=1)
            keep[1:] = ~dup
        return P[keep]
    
    def _clip_polyline_with_latlon_by_x_band(
        self,
        X: np.ndarray,         # (N,2) local-frame (x,y)
        lat: np.ndarray,       # (N,)
        lon: np.ndarray,       # (N,)
        band: Tuple[float, float]
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Returns (X_clip, lat_clip, lon_clip), each with >=2 points if band intersects.
        Boundary points are linearly interpolated (for both XY and lat/lon).
        """
        x_lo, x_hi = band
        x = X[:, 0]
        pts_xy, pts_lat, pts_lon = [], [], []

        for j in range(len(X) - 1):
            x0, x1 = x[j], x[j + 1]
            P0, P1 = X[j], X[j + 1]
            lat0, lat1 = lat[j], lat[j + 1]
            lon0, lon1 = lon[j], lon[j + 1]

            seg_min, seg_max = (x0, x1) if x0 <= x1 else (x1, x0)
            if seg_max < x_lo or seg_min > x_hi:
                continue  # segment entirely outside band

            # keep start if inside
            if x_lo <= x0 <= x_hi:
                pts_xy.append(P0); pts_lat.append(lat0); pts_lon.append(lon0)

            # intersections with x_lo and x_hi (strict crossings)
            for xb in (x_lo, x_hi):
                denom = (x1 - x0)
                if denom != 0.0 and (x0 - xb) * (x1 - xb) < 0.0:
                    t = (xb - x0) / denom
                    pts_xy.append(P0 + t * (P1 - P0))
                    pts_lat.append(lat0 + t * (lat1 - lat0))
                    pts_lon.append(lon0 + t * (lon1 - lon0))

            # keep end if last seg and inside
            if j == len(X) - 2 and (x_lo <= x1 <= x_hi):
                pts_xy.append(P1); pts_lat.append(lat1); pts_lon.append(lon1)

        if not pts_xy:
            return np.empty((0, 2)), np.empty((0,)), np.empty((0,))

        XYc = np.vstack(pts_xy)
        latc = np.asarray(pts_lat, dtype=float)
        lonc = np.asarray(pts_lon, dtype=float)

        # drop adjacent duplicates (numeric noise)
        if len(XYc) > 1:
            dup = np.all(np.isclose(np.diff(XYc, axis=0), 0.0, atol=1e-9), axis=1)
            keep = np.ones(len(XYc), dtype=bool); keep[1:] = ~dup
            XYc, latc, lonc = XYc[keep], latc[keep], lonc[keep]
        return XYc, latc, lonc
    
    def _arclength(self, X: np.ndarray) -> np.ndarray:
        """Cumulative arclength for 2D polyline (x,y)."""
        d = np.hypot(np.diff(X[:,0]), np.diff(X[:,1]))
        return np.concatenate([[0.0], np.cumsum(d)])
    
    def _interp_by_arclength(self, X: np.ndarray, s_targets: np.ndarray) -> np.ndarray:
        """
        Interpolate 2D polyline X to arclength grid s_targets. Returns Mx2 array.
        """
        s = self._arclength(X)
        keep = np.concatenate([[True], np.diff(s) > 1e-9])  # drop zero-length steps
        s, X = s[keep], X[keep]
        xi = np.interp(s_targets, s, X[:,0])
        yi = np.interp(s_targets, s, X[:,1])
        return np.column_stack([xi, yi])
    
    def _interp_attr_by_arclength(
        self,
        X: np.ndarray,          # (N,2) polyline
        attr: np.ndarray,       # (N,) attribute values (lat or lon)
        s_targets: np.ndarray   # (M,) arclength positions
    ) -> np.ndarray:
        s = self._arclength(X)
        keep = np.concatenate([[True], np.diff(s) > 1e-9])
        s, Xattr = s[keep], attr[keep]
        return np.interp(s_targets, s, Xattr)
    
    def _spacing_from_overlap(self,
        Xi_seg: np.ndarray, Xk_seg: np.ndarray,
        step_ft: Optional[int], n_samples: Optional[int]
    ) -> Tuple[float, int, float]:
        """
        Return (horizontal_mean, n_samples_used, overlap_len_ft_used).
        Overlap length used = min(arclength_i, arclength_k).
        """
        si = self._arclength(Xi_seg); Li = si[-1]
        sk = self._arclength(Xk_seg); Lk = sk[-1]
        Lmin = max(min(Li, Lk), 1e-6)

        if n_samples is None:
            step = max(int(step_ft or 100), 1)
            n = max(int(np.floor(Lmin / step)) + 1, 2)  # works even if overlap < step
        else:
            n = max(int(n_samples), 2)

        t = np.linspace(0.0, 1.0, n)
        Pi = self._interp_by_arclength(Xi_seg, t * Li)
        Pk = self._interp_by_arclength(Xk_seg, t * Lk)

        # Align-Y rule: use Δx only (equivalent to "set y_k := y_i")
        dx = np.abs(Pk[:,0] - Pi[:,0])
        horizontal_mean = float(dx.mean())
        return horizontal_mean, int(n), float(Lmin)
    
    def _bin8_from_azimuth_deg(self, az: np.ndarray) -> np.ndarray:
        # Normalize to [-180, 180)
        a = (az + 180.0) % 360.0 - 180.0
        # Shift so 0° (E) center → bins of 45°
        # Index: round(a / 45) mod 8, but careful with boundaries
        idx = np.floor(((a + 22.5) % 360.0) / 45.0).astype(int)  # 0..7
        return self._DIR8_LABELS[idx]
    
    def _modal_direction_geodetic_over_overlap(
        self,
        Xi_seg: np.ndarray, Xk_seg: np.ndarray,          # local XY clipped segments
        lat_i_seg: np.ndarray, lon_i_seg: np.ndarray,    # clipped lat/lon for i
        lat_k_seg: np.ndarray, lon_k_seg: np.ndarray,    # clipped lat/lon for k
        step_ft: Optional[int], n_samples: Optional[int]
    ) -> Tuple[str, float, str]:
        """
        Returns (modal_direction_8way, confidence, distribution_string)
        using geodetic bearings at the same sample grid as spacing.
        """
        # Sample count (match spacing logic)
        si = self._arclength(Xi_seg); Li = si[-1]
        sk = self._arclength(Xk_seg); Lk = sk[-1]
        Lmin = max(min(Li, Lk), 1e-6)

        if n_samples is None:
            step = max(int(step_ft or 100), 1)
            n = max(int(np.floor(Lmin / step)) + 1, 2)
        else:
            n = max(int(n_samples), 2)
        t = np.linspace(0.0, 1.0, n)

        # Interpolate lat/lon by arclength within each clipped segment
        s_i = t * Li
        s_k = t * Lk
        lat_i = self._interp_attr_by_arclength(Xi_seg, lat_i_seg, s_i)
        lon_i = self._interp_attr_by_arclength(Xi_seg, lon_i_seg, s_i)
        lat_k = self._interp_attr_by_arclength(Xk_seg, lat_k_seg, s_k)
        lon_k = self._interp_attr_by_arclength(Xk_seg, lon_k_seg, s_k)

        # Geodetic forward azimuth i→k at each sample
        geod = Geod(ellps="WGS84")
        az12, _, _ = geod.inv(lon_i, lat_i, lon_k, lat_k)  # degrees, vectorized

        # 8-way bins + aggregation
        labels = self._bin8_from_azimuth_deg(az12)
        # mode + confidence
        uniq, counts = np.unique(labels, return_counts=True)
        best_idx = np.argmax(counts)
        mode = uniq[best_idx]
        conf = counts[best_idx] / float(n)

        # compact distribution string for QA
        parts = [f"{u}:{c/float(n):.2f}" for u, c in sorted(zip(uniq, counts), key=lambda z: -z[1])]
        dist_str = ",".join(parts)
        return mode, conf, dist_str
    
    def _process_batch(
        self,
        i_idx: np.ndarray,
        k_idx: np.ndarray,
        ids: np.ndarray,
        coords: np.ndarray,        # midpoint coords [["x","y","tvd"]]
        directions: np.ndarray,    # drill dirs per well (EW/NS)
        *,
        step_ft: int,
        n_samples: Optional[int],
        max_crossline_ft: Optional[float],
        crossline_percentile: float,
        ds_crossline_step_ft: int,
        emit_rejected: bool,
        use_pca_axis: bool
    ) -> pd.DataFrame:
        """
        Vectorizes cross-line precheck per 'i'; loops only over survivor pairs
        for exact clip+sample+geodetic modal direction.
        Rejected rows have all computed metrics filled with NaN.
        """
        rows: List[Dict] = []
        cache = getattr(self, "_paircache", None)
        if cache is None or cache.get("use_pca_axis", None) != use_pca_axis:
            # Build cache once (local frames + coarse resamples)
            self._build_pair_cache(use_pca_axis, ds_crossline_step_ft)

        # Group pairs by source index i
        for i in np.unique(i_idx):
            mask_i = (i_idx == i)
            k_list = k_idx[mask_i]
            if k_list.size == 0:
                continue

            uwi_i = ids[i]
            origin_i = self._paircache["origin"][uwi_i]
            ex_i = self._paircache["ex"][uwi_i]
            ey_i = self._paircache["ey"][uwi_i]

            # Coarse arrays (vectorized across all neighbors k of this i)
            Pi_coarse = self._paircache["XY_coarse"][uwi_i]                               # (M_ds, 2)
            Pk_coarse = np.stack([self._paircache["XY_coarse"][ids[k]] for k in k_list])  # (K, M_ds, 2)

            # Project coarse to i's local frame
            Ri = Pi_coarse - origin_i                                                    # (M_ds, 2)
            Rk = Pk_coarse - origin_i[None, None, :]                                     # (K, M_ds, 2)
            xi = Ri @ ex_i                                                               # (M_ds,)
            yi = Ri @ ey_i                                                               # (M_ds,)
            xk = np.einsum("kmd,d->km", Rk, ex_i)                                        # (K, M_ds)
            yk = np.einsum("kmd,d->km", Rk, ey_i)                                        # (K, M_ds)

            # x-overlap bands per k (vectorized)
            xi_min, xi_max = float(xi.min()), float(xi.max())
            xk_min = xk.min(axis=1)                                                      # (K,)
            xk_max = xk.max(axis=1)                                                      # (K,)
            x_lo = np.maximum(xi_min, xk_min)                                            # (K,)
            x_hi = np.minimum(xi_max, xk_max)                                            # (K,)
            has_overlap = x_hi > x_lo                                                    # (K,)

            # Emit 'no_overlap_x' rejections (NaNs for metrics) and continue if none remain
            if not has_overlap.any():
                if emit_rejected:
                    for k in k_list:
                        rows.append({
                            "well_i": uwi_i, "well_k": ids[k],
                            "horizontal_dist": np.nan, "vertical_dist": np.nan, "3D_dist": np.nan,
                            "drill_direction_i": directions[i], "drill_direction_k": directions[k],
                            "direction_to_k_from_i": np.nan, "direction_confidence": np.nan,
                            "direction_distribution": "",
                            "overlap_len_ft": np.nan, "n_samples": np.nan, "dy_p5": np.nan,
                            "reject_reason": "no_overlap_x"
                        })
                continue

            # Masks for 'inside band' (vectorized across k)
            mask_kj = (xk >= x_lo[:, None]) & (xk <= x_hi[:, None])                      # (K, M_ds)
            mask_km = (xi[None, :] >= x_lo[:, None]) & (xi[None, :] <= x_hi[:, None])    # (K, M_ds)

            # Downsampled cross-line guardrail (robust "min" via percentile)
            # Build (K, M_ds, M_ds) of |yk(k,j) - yi(m)|; mask outside band; min over j; percentile over m
            YK = yk[:, :, None]                                                          # (K, M_ds, 1)
            YI = yi[None, None, :]                                                       # (1, 1, M_ds)
            D = np.abs(YK - YI)                                                          # (K, M_ds, M_ds)
            mask_grid = mask_kj[:, :, None] & mask_km[:, None, :]                        # (K, M_ds, M_ds)
            D[~mask_grid] = np.inf

            Dmin_km = np.min(D, axis=1)                                                  # (K, M_ds)
            with np.errstate(invalid="ignore"):
                Dmin_km = np.where(np.isfinite(Dmin_km), Dmin_km, np.nan)
                dy_p = np.nanpercentile(Dmin_km, crossline_percentile, axis=1)           # (K,)

            keep = has_overlap.copy()
            if max_crossline_ft is not None:
                keep &= np.isfinite(dy_p) & (dy_p <= max_crossline_ft)

            # Emit 'too_crossline' rejections (NaNs for metrics)
            if emit_rejected:
                rej = (~keep) & has_overlap
                for idx, k in enumerate(k_list):
                    if rej[idx]:
                        rows.append({
                            "well_i": uwi_i, "well_k": ids[k],
                            "horizontal_dist": np.nan, "vertical_dist": np.nan, "3D_dist": np.nan,
                            "drill_direction_i": directions[i], "drill_direction_k": directions[k],
                            "direction_to_k_from_i": np.nan, "direction_confidence": np.nan,
                            "direction_distribution": np.nan,
                            "overlap_len_ft": np.nan, "n_samples": np.nan, "dy_p5": np.nan,
                            "reject_reason": "too_crossline"
                        })

            # Nothing survives → next i
            if not keep.any():
                continue

            # Project full-resolution i once (for exact spacing & direction)
            df_i = self.trajectories[uwi_i].sort_values("md")
            Xi_full = self._project_xy_to_frame(df_i, origin_i, ex_i, ey_i)
            lat_i_full = df_i["latitude"].to_numpy(dtype=float)
            lon_i_full = df_i["longitude"].to_numpy(dtype=float)

            # Process survivors (small loop)
            for idx, k in enumerate(k_list):
                if not keep[idx]:
                    continue

                uwi_k = ids[k]
                band = (float(x_lo[idx]), float(x_hi[idx]))

                # Full-resolution projection for k
                df_k = self.trajectories[uwi_k].sort_values("md")
                Xk_full = self._project_xy_to_frame(df_k, origin_i, ex_i, ey_i)
                lat_k_full = df_k["latitude"].to_numpy(dtype=float)
                lon_k_full = df_k["longitude"].to_numpy(dtype=float)

                # Exact clip of XY + lat/lon (supports <100' overlaps)
                Xi_seg = self._clip_polyline_by_x_band(Xi_full, band)
                Xk_seg = self._clip_polyline_by_x_band(Xk_full, band)
                if Xi_seg.shape[0] == 0 or Xk_seg.shape[0] == 0:
                    # Extremely rare numeric edge; treat as no-overlap
                    if emit_rejected:
                        rows.append({
                            "well_i": uwi_i, "well_k": uwi_k,
                            "horizontal_dist": np.nan, "vertical_dist": np.nan, "3D_dist": np.nan,
                            "drill_direction_i": directions[i], "drill_direction_k": directions[k],
                            "direction_to_k_from_i": np.nan, "direction_confidence": np.nan,
                            "direction_distribution": np.nan,
                            "overlap_len_ft": np.nan, "n_samples": np.nan, "dy_p5": np.nan,
                            "reject_reason": "no_overlap_x"
                        })
                    continue

                Xi_seg_ll, lat_i_seg, lon_i_seg = self._clip_polyline_with_latlon_by_x_band(
                    Xi_full, lat_i_full, lon_i_full, band
                )
                Xk_seg_ll, lat_k_seg, lon_k_seg = self._clip_polyline_with_latlon_by_x_band(
                    Xk_full, lat_k_full, lon_k_full, band
                )

                # Spacing over overlap (align-Y → mean |Δx|)
                horiz_mean, n_used, Lmin = self._spacing_from_overlap(
                    Xi_seg, Xk_seg, step_ft=step_ft, n_samples=n_samples
                )

                # Geodetic modal direction over same samples
                dir_mode, dir_conf, dir_dist = self._modal_direction_geodetic_over_overlap(
                    Xi_seg, Xk_seg, lat_i_seg, lon_i_seg, lat_k_seg, lon_k_seg,
                    step_ft=step_ft, n_samples=n_samples
                )

                # Vertical from midpoints
                A = coords[i]; B = coords[k]
                vertical = float(abs(B[2] - A[2]))
                dist3d = float(np.hypot(horiz_mean, vertical))

                rows.append({
                    "well_i": uwi_i, "well_k": uwi_k,
                    "horizontal_dist": horiz_mean, "vertical_dist": vertical, "3D_dist": dist3d,
                    "drill_direction_i": directions[i], "drill_direction_k": directions[k],
                    "direction_to_k_from_i": dir_mode,
                    "direction_confidence": dir_conf,
                    "direction_distribution": dir_dist,
                    "overlap_len_ft": Lmin, "n_samples": n_used,
                    "dy_p5": float(dy_p[idx]) if np.isfinite(dy_p[idx]) else np.nan,
                    "reject_reason": ""
                })

        return pd.DataFrame(rows)

## 2. Loading Header and GeoSurvey either from Excel/csv/SQL into Pandas DataFrame

In [3]:
loader = WellDataLoader(db = DatabricksOdbcConnector(), 
                        log_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\logs")

In [4]:
df_MB_header = loader.get_header_data(basin="MB", start_year=2014)

[WellDataLoaderLogger] INFO (08-15 02:13 PM): Loading header data from SQL. (Line: 84) [well_data_manager.py]

c:\users\apoorva.saxena\onedrive - sitio royalties\desktop\project - apoorva\python\parent_child_spacing\src\utils\database_manager.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(sql_query, self.connection)


In [5]:
df_MB_directional = loader.get_directional_data()

[WellDataLoaderLogger] INFO (08-15 02:13 PM): Loading directional data from SQL. (Line: 104) [well_data_manager.py]



In [6]:
processor = GeoSurveyProcessor(log_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\logs")

[GeoLogger] INFO (08-15 02:15 PM): GeoSurveyProcessor initialized. (Line: 186) [well_data_manager.py]



In [7]:
df_utm = processor.compute_utm_coordinates(df=df_MB_directional)

[GeoLogger] INFO (08-15 02:15 PM): ✅ Using lat/lon from input DataFrame. (Line: 262) [well_data_manager.py]

[GeoLogger] INFO (08-15 02:15 PM): ✅ UTM coordinate computation complete in 3.70 sec. (Line: 317) [well_data_manager.py]



In [8]:
df_utm_lateral = processor.filter_after_heel_point(df=df_utm)

In [10]:
spacing_calculator = WellSpacingCalculator(trajectories=df_utm_lateral)

In [11]:
spacing_calculator._calculate_spacing_statistics(
    max_distance_miles=2.0,
    use_interpolation=False,
    step_ft=100,
    n_samples=None,
    max_crossline_ft=2000.0,
    crossline_percentile=5.0,
    ds_crossline_step_ft=200,
    emit_rejected=True,
    use_pca_axis=True,
    batch_size=500_000,
    save_batches_dir="spacing_batches_MB"
)

🚀 Calculating Spacing (Parallel): |████████████████████████████████████████| 100% 4/4 [00:00<00:00]


✅ All batches saved to spacing_batches_MB


In [12]:
df_spacing_new = spacing_calculator._load_saved_batches(batch_folder="spacing_batches_MB") # Loading the saved batches into a DataFrame

🔍 Found 4 batch files. Loading and combining...
✅ Loaded 1,415,061 rows from all batches.


In [15]:
df_spacing_new

,well_i,well_k,horizontal_dist,vertical_dist,3D_dist,drill_direction_i,drill_direction_k,direction_to_k_from_i,direction_confidence,direction_distribution,overlap_len_ft,n_samples,dy_p5,reject_reason
0,42003442780000,42003458480000,NaN,NaN,NaN,NS,NS,None,NaN,None,NaN,NaN,NaN,too_crossline
1,42003442780000,42003459930000,NaN,NaN,NaN,NS,NS,None,NaN,None,NaN,NaN,NaN,too_crossline
2,42003442780000,42003469280000,NaN,NaN,NaN,NS,NS,None,NaN,None,NaN,NaN,NaN,too_crossline
3,42003442780000,42003469980000,NaN,NaN,NaN,NS,NS,None,NaN,None,NaN,NaN,NaN,too_crossline
4,42003452020000,42003461880000,NaN,NaN,NaN,NS,NS,None,NaN,None,NaN,NaN,NaN,too_crossline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1415056,42501376030000,42501373560000,2439.776162,10.7820,2439.799986,NS,NS,E,0.375000,"E:0.38,W:0.35,SW:0.12,SE:0.10,S:0.04",4778.374656,48.0,377.378599,
1415057,42501376030000,42501373650000,671.556512,8.5280,671.610658,NS,NS,S,0.384615,"S:0.38,SE:0.31,SW:0.31",1247.280835,13.0,1029.327461,
1415058,42501376030000,42501373660000,663.285448,20.7925,663.611267,NS,NS,SW,0.384615,"SW:0.38,SE:0.31,E:0.15,S:0.08,W:0.08",1231.994713,13.0,389.713407,
1415059,42501376030000,42501375980000,0.124115,1.8600,1.864136,NS,NS,N,1.000000,N:1.00,5032.761191,51.0,1957.358225,


In [22]:
df_spacing_new[df_spacing_new['well_i']=='42003472060000']

,well_i,well_k,horizontal_dist,vertical_dist,3D_dist,drill_direction_i,drill_direction_k,direction_to_k_from_i,direction_confidence,direction_distribution,overlap_len_ft,n_samples,dy_p5,reject_reason
4799,42003472060000,42003473240000,NaN,NaN,NaN,NS,NS,None,NaN,None,NaN,NaN,NaN,too_crossline
4800,42003472060000,42003476440000,NaN,NaN,NaN,NS,NS,None,NaN,None,NaN,NaN,NaN,too_crossline
4801,42003472060000,42003452020000,0.157984,366.670,366.670034,NS,NS,N,1.000000,N:1.00,4295.000899,43.0,1354.576878,
4802,42003472060000,42003461880000,0.266119,331.485,331.485107,NS,NS,S,1.000000,S:1.00,4276.066947,43.0,1210.379304,
4803,42003472060000,42003472070000,0.970277,51.570,51.579127,NS,NS,S,1.000000,S:1.00,4428.348670,45.0,1214.617774,
4804,42003472060000,42003473970000,2239.027268,146.485,2243.813932,NS,NS,NW,0.409091,"NW:0.41,E:0.39,N:0.09,NE:0.09,W:0.02",4377.109958,44.0,648.906228,


In [26]:
df_utm_lateral[df_utm_lateral['uwi'].isin(['42003472060000','42003452020000'])]

,uwi,md,tvd,inclination,azimuth,latitude,longitude,deviation_E/W,E/W,deviation_N/S,N/S,point_type_name,x,y,utm_zone,epsg_code,z
99,42003452020000,6725.0,6393.54,80.8,345.4,32.093193,-102.723155,155.41,WEST,623.21,NORTH,80 DEGREE HEEL POINT,2.345365e+06,1.165694e+07,13,EPSG:32613,-6393.54
100,42003452020000,6756.0,6397.80,83.4,346.7,32.093274,-102.723179,162.81,WEST,653.01,NORTH,None,2.345357e+06,1.165697e+07,13,EPSG:32613,-6397.80
101,42003452020000,6810.0,6402.22,87.2,347.7,32.093418,-102.723218,174.73,WEST,705.48,NORTH,None,2.345344e+06,1.165702e+07,13,EPSG:32613,-6402.22
102,42003452020000,6873.0,6405.57,86.7,346.7,32.093586,-102.723263,188.67,WEST,766.82,NORTH,None,2.345328e+06,1.165708e+07,13,EPSG:32613,-6405.57
103,42003452020000,6937.0,6408.70,87.7,346.0,32.093756,-102.723312,203.75,WEST,828.94,NORTH,None,2.345312e+06,1.165714e+07,13,EPSG:32613,-6408.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14976,42003472060000,11249.0,6751.17,89.7,346.7,32.102988,-102.730654,595.31,WEST,5107.64,NORTH,None,2.342967e+06,1.166045e+07,13,EPSG:32613,-6751.17
14977,42003472060000,11339.0,6751.01,90.5,346.7,32.103228,-102.730721,616.01,WEST,5195.23,NORTH,None,2.342945e+06,1.166054e+07,13,EPSG:32613,-6751.01
14978,42003472060000,11429.0,6749.36,91.6,346.4,32.103468,-102.730789,636.94,WEST,5282.74,NORTH,None,2.342922e+06,1.166063e+07,13,EPSG:32613,-6749.36
14979,42003472060000,11468.0,6748.10,92.1,346.3,32.103571,-102.730819,646.14,WEST,5320.62,NORTH,None,2.342912e+06,1.166066e+07,13,EPSG:32613,-6748.10
